# Portable TRM GEMM Refiner

This notebook is the main entrypoint for running the Turing-first TRM GEMM prototype.

Recommended runtime:
- Google Colab
- GPU: `T4`

What it does:
- installs the package
- checks backend/runtime mode
- runs the Colab smoke test
- generates offline traces
- builds the tiny recursive model
- runs a short rollout


In [6]:
!git clone https://github.com/Cree0618/trm-youtubevids.git


Cloning into 'trm-youtubevids'...
remote: Enumerating objects: 41, done.
remote: Counting objects: 100% (41/41), done.
remote: Compressing objects: 100% (34/34), done.
remote: Total 41 (delta 4), reused 41 (delta 4), pack-reused 0 (from 0)
Receiving objects: 100% (41/41), 88.05 KiB | 733.00 KiB/s, done.
Resolving deltas: 100% (4/4), done.


In [7]:
!ls



sample_data  trm-youtubevids


In [8]:
# If running in Colab, mount or clone the repo first, then set the project root below.
import os
from pathlib import Path

PROJECT_ROOT = Path("/content/trm-youtubevids")
if not PROJECT_ROOT.exists():
    PROJECT_ROOT = Path.cwd()

print('PROJECT_ROOT =', PROJECT_ROOT)
os.chdir(PROJECT_ROOT)

PROJECT_ROOT = /content/trm-youtubevids


In [9]:
!pip install -e .
!pip install -r requirements-colab.txt

Obtaining file:///content/trm-youtubevids
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for trm-gemm (pyproject.toml) ... done
  Created wheel for trm-gemm: filename=trm_gemm-0.1.0-0.editable-py3-none-any.whl size=2708 sha256=52ce84f4bf912b6f6c6c3d568574c66659944f5c08ff4baa5fbaae0841d1d2a3
  Stored in directory: /tmp/pip-ephem-wheel-cache-wb98johr/wheels/7c/10/ad/aa3047aeaf15cddb5bd07bf1c493df647fa66a18a67c001568
Successfully built trm-gemm


In [10]:
!python -m trm_gemm.colab_smoke

{
  "backend_mode": "triton_cuda",
  "cuda_available": true,
  "device_name": "Tesla T4",
  "triton_available": true
}
generated_trace_records=18
losses {'edit': 1455.9766, 'feasibility': 8.8646, 'value': 106960.0469, 'halt': 871.2732, 'total': 28422.6719}
random_runtime_us 175.7844
greedy_runtime_us 167.2475
rollout_edits [{'field_name': 'NUM_STAGES', 'value': 2}, {'field_name': 'VEC_A', 'value': 1}, {'field_name': 'VEC_A', 'value': 4}]
done


In [11]:
import torch

from trm_gemm.backend import TritonGemmBackend
from trm_gemm.baselines import greedy_search, random_search
from trm_gemm.data import TraceDataset, TraceGenerationConfig, generate_trace_records
from trm_gemm.model import TinyRecursiveGemmRefiner, rollout_refiner
from trm_gemm.training import compute_losses
from trm_gemm.types import GemmTaskSpec, T4

backend = TritonGemmBackend()
print({
    'backend_mode': backend.mode,
    'cuda_available': torch.cuda.is_available(),
    'device_name': torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    'triton_available': backend.capabilities.triton_available,
})

{'backend_mode': 'triton_cuda', 'cuda_available': True, 'device_name': 'Tesla T4', 'triton_available': True}


In [12]:
tasks = [
    GemmTaskSpec(512, 512, 512),
    GemmTaskSpec(1024, 512, 768),
    GemmTaskSpec(1536, 1024, 640),
    GemmTaskSpec(2048, 1024, 1024),
]

records = generate_trace_records(
    tasks,
    T4,
    backend,
    TraceGenerationConfig(seeds_per_task=2, max_steps_per_seed=3),
)

dataset = TraceDataset(records)
print('num_records =', len(dataset))
print('first_record =', dataset[0].to_dict())

num_records = 32
first_record = {'task': {'m': 512, 'n': 512, 'k': 512, 'dtype_a': 'fp32', 'dtype_b': 'fp32', 'dtype_acc': 'fp32', 'dtype_out': 'fp32', 'layout_a': 'row_major', 'layout_b': 'col_major', 'layout_c': 'row_major'}, 'gpu_target': {'arch_family': 'turing', 'compute_capability': '7.5', 'max_shared_mem_bytes': 65536, 'max_registers_per_thread': 255, 'warp_size': 32, 'tensor_cores': True, 'supported_dtypes': ['fp32', 'fp16'], 'instruction_families': ['simt', 'tensorcore'], 'arch_extensions': {}}, 'state_t': {'portable_core': {'BLOCK_M': 64, 'BLOCK_N': 64, 'BLOCK_K': 32, 'NUM_WARPS': 4, 'NUM_STAGES': 3, 'GROUP_M': 1, 'SPLIT_K': 4, 'VEC_A': 1, 'VEC_B': 1}, 'arch_extensions': {}}, 'feedback_t': {'compiled': True, 'correct': True, 'runtime_us': 281.5203828873207, 'normalized_tflops': 0.0009535204991087343, 'registers_per_thread': 44, 'shared_mem_bytes': 49152, 'occupancy': 0.345632798573975, 'failure_reason': 'valid'}, 'edit_t': {'field_name': 'SPLIT_K', 'value': 1}, 'state_t1': {'

In [13]:
model = TinyRecursiveGemmRefiner()
batch = records[: min(8, len(records))]
losses = compute_losses(model, batch)
print({k: round(v.item(), 4) for k, v in losses.items()})

{'edit': 3720665.0, 'feasibility': 1779980.375, 'value': 2.4898136727302963e+17, 'halt': 803075.625, 'total': 6.224534181825741e+16}


In [14]:
task = tasks[0]
random_schedule, random_feedback = random_search(backend, task, T4, budget=6, seed=0)
greedy_schedule, greedy_feedback = greedy_search(backend, task, T4, budget=6)

print('random_schedule =', random_schedule.to_dict())
print('random_feedback =', random_feedback.to_dict())
print('greedy_schedule =', greedy_schedule.to_dict())
print('greedy_feedback =', greedy_feedback.to_dict())

random_schedule = {'portable_core': {'BLOCK_M': 64, 'BLOCK_N': 64, 'BLOCK_K': 32, 'NUM_WARPS': 4, 'NUM_STAGES': 1, 'GROUP_M': 4, 'SPLIT_K': 1, 'VEC_A': 4, 'VEC_B': 1}, 'arch_extensions': {}}
random_feedback = {'compiled': True, 'correct': True, 'runtime_us': 175.78439140993655, 'normalized_tflops': 0.0015270721925133687, 'registers_per_thread': 33, 'shared_mem_bytes': 16384, 'occupancy': 0.8433155080213903, 'failure_reason': 'valid'}
greedy_schedule = {'portable_core': {'BLOCK_M': 64, 'BLOCK_N': 64, 'BLOCK_K': 16, 'NUM_WARPS': 4, 'NUM_STAGES': 1, 'GROUP_M': 4, 'SPLIT_K': 1, 'VEC_A': 4, 'VEC_B': 4}, 'arch_extensions': {}}
greedy_feedback = {'compiled': True, 'correct': True, 'runtime_us': 167.2475317340256, 'normalized_tflops': 0.001605018939393939, 'registers_per_thread': 34, 'shared_mem_bytes': 8192, 'occupancy': 0.953030303030303, 'failure_reason': 'valid'}


In [15]:
edits = rollout_refiner(model, task, T4, random_schedule, random_feedback, steps=4)
print('rollout_edits =', [edit.to_dict() for edit in edits])

rollout_edits = [{'field_name': 'NUM_STAGES', 'value': 2}, {'field_name': 'VEC_A', 'value': 2}, {'field_name': 'BLOCK_N', 'value': 128}, {'field_name': 'BLOCK_N', 'value': 64}]


In [16]:
dataset_path = PROJECT_ROOT / 'tmp_trm_gemm_traces.jsonl'
dataset.to_jsonl(dataset_path)
print('saved', dataset_path)

saved /content/trm-youtubevids/tmp_trm_gemm_traces.jsonl


In [17]:
!cat {dataset_path} | head -3

{"task": {"m": 512, "n": 512, "k": 512, "dtype_a": "fp32", "dtype_b": "fp32", "dtype_acc": "fp32", "dtype_out": "fp32", "layout_a": "row_major", "layout_b": "col_major", "layout_c": "row_major"}, "gpu_target": {"arch_family": "turing", "compute_capability": "7.5", "max_shared_mem_bytes": 65536, "max_registers_per_thread": 255, "warp_size": 32, "tensor_cores": true, "supported_dtypes": ["fp32", "fp16"], "instruction_families": ["simt", "tensorcore"], "arch_extensions": {}}, "state_t": {"portable_core": {"BLOCK_M": 64, "BLOCK_N": 64, "BLOCK_K": 32, "NUM_WARPS": 4, "NUM_STAGES": 3, "GROUP_M": 1, "SPLIT_K": 4, "VEC_A": 1, "VEC_B": 1}, "arch_extensions": {}}, "feedback_t": {"compiled": true, "correct": true, "runtime_us": 281.5203828873207, "normalized_tflops": 0.0009535204991087343, "registers_per_thread": 44, "shared_mem_bytes": 49152, "occupancy": 0.345632798573975, "failure_reason": "valid"}, "edit_t": {"field_name": "SPLIT_K", "value": 1}, "state_t1": {"portable_core": {"BLOCK_M": 64, 